# Reward-gap clusters: inspect the prompts and answers

**Upload this single notebook into your existing `reward_gap_followup` RunPod folder**, beside `RUN_FOLLOWUP.ipynb`. Open it with a fresh Python kernel and choose **Run → Run All Cells**.

It uses saved hidden-state vectors and saved proxy/judge scores. **No PPO, model loading, teacher calls, HF token, or GPU is required.** CPU/RAM and the completed project files are sufficient. It writes only to a new `cluster_visualization` folder.

The result is a clickable, searchable explorer: select a cluster or a point, read the full conversation and answer, compare the actual gap, and add manual notes or provisional cluster labels. You can export everything and open the interactive HTML on your own computer.

**Two different k values:** `N_CLUSTERS=20` controls the descriptive k-means groups. `NEIGHBORS=31` controls the separate kNN geometry check. Clusters are fit on the original 1,024-dimensional L2-normalized vectors, while PCA is used only for the 2D map.

**Interpretation:** high gap means $z_P-z_J>\theta$, not a human-confirmed reward hack. Ordinary/non-high gaps are not automatically good answers. The notebook helps you investigate possible behaviors, not assign verified failure types automatically.

## 1. Paths and settings

Defaults find your project and its latest saved study. Set `PROJECT_DIR` explicitly if the folder has a different name. Set `STUDY_DIR` only to inspect a particular `outputs/study_...` directory. You can also use an **extracted** results archive; if it lacks `candidate_bank.csv`, the notebook uses saved development-fit vectors as its reference instead.

By default, original memory + saved development and refresh vectors are included. Final PPO answer tables without saved vectors are listed as unavailable. Nothing is silently re-embedded. The old datastore is not edited. Scores retain their original source protocol, including any earlier token limit; use source filters when comparing the old memory with later whole-answer scores.

Wait until your main study is complete if you want a stable snapshot. Only completed, checksum-verified vector exports are read.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib.util

PROJECT_DIR = None        # Example: "/workspace/reward_gap_followup"
STUDY_DIR = None          # Example: "/workspace/reward_gap_followup/outputs/study_..."
BANK_CSV = None           # Only needed if candidate_bank.csv lives outside the project.
OUTPUT_DIR = None         # Default: PROJECT_DIR / "cluster_visualization"

N_CLUSTERS = 20
NEIGHBORS = 31
TEMPERATURE = 0.05
SEED = 42
CPU_THREADS = 4
GEOMETRY_QUERIES = 1200   # Uses all 1,024 offline-test responses when available.
BOOTSTRAP_DRAWS = 1000
EMBED_EXPLORER = True     # Large HTML output; set False to use the standalone file only.

# Existing compatible packages are kept. Installation happens only if missing.
requirements = {"numpy":"numpy>=1.26,<3", "pandas":"pandas>=2.2,<4",
                "sklearn":"scikit-learn>=1.5,<2", "scipy":"scipy>=1.11,<2",
                "matplotlib":"matplotlib>=3.8,<4", "threadpoolctl":"threadpoolctl>=3.1,<4"}
missing = [package for module,package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
print("CPU analysis requirements ready. No torch/transformers changes are made.")

## 2. Analysis functions and offline explorer

Run both cells below (Run All does this automatically). Their source is collapsed for readability. All required code is embedded in this notebook; no helper Python file needs to be uploaded.

In [ ]:
"""Standalone saved-vector exploration. No reward models, PPO, or network calls."""
from pathlib import Path
import hashlib, json, re, unicodedata, time, zipfile
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, spearmanr
from threadpoolctl import threadpool_limits
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


def digest_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(2**20),b''):h.update(b)
    return h.hexdigest()


def digest_json(x):return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False,allow_nan=False).encode()).hexdigest()

def read_json(p):return json.loads(Path(p).read_text(encoding='utf-8'))

def write_json(p,x):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(json.dumps(x,ensure_ascii=False,indent=2,allow_nan=False)+'\n',encoding='utf-8')


def conversation_group(prompt):
    # Use the first user turn so later turns/alternative answers cannot become neighbors.
    text=str(prompt).strip()
    matches=list(re.finditer(r'(?:\A|(?:\r?\n){2,})[ \t]*(Human|Assistant):[ \t]*',text))
    if matches and matches[0].group(1)=='Human' and not text[:matches[0].start()].strip():
        text=text[matches[0].end():matches[1].start() if len(matches)>1 else len(text)]
    return digest_json(' '.join(unicodedata.normalize('NFKC',text).casefold().split()))


def unit_vectors(x):
    x=np.asarray(x,dtype=np.float32)
    if x.ndim!=2 or not len(x) or not np.isfinite(x).all():raise ValueError('Expected a nonempty finite vector matrix.')
    norms=np.linalg.norm(x,axis=1)
    if (norms<1e-12).any():raise ValueError('Zero-length vectors cannot be clustered.')
    if not np.allclose(norms,1,atol=.01):print('L2-normalizing stored vectors for analysis.')
    return np.ascontiguousarray(x/norms[:,None])


def resolve_paths(project_dir=None,study_dir=None):
    if project_dir:
        project=Path(project_dir).expanduser().resolve()
        if not project.is_dir():raise FileNotFoundError(f'PROJECT_DIR does not exist: {project}')
    else:
        candidates=[Path.cwd(),Path.cwd()/'reward_gap_followup',Path('/workspace/reward_gap_followup'),
                    Path.cwd()/'reward_gap_overnight',Path('/workspace/reward_gap_overnight')]
        project=next((p.resolve() for p in candidates if (p/'inputs/memory/detectors/gap_knn.npz').is_file()),None)
        if project is None:raise FileNotFoundError('Set PROJECT_DIR to the extracted reward_gap_followup project, or an extracted results folder containing inputs/memory.')
    if study_dir:
        study=Path(study_dir).expanduser().resolve()
        if not study.is_dir():raise FileNotFoundError(f'STUDY_DIR does not exist: {study}')
    elif (project/'outputs/latest.json').is_file():
        study=(project/read_json(project/'outputs/latest.json')['relative_output']).resolve()
    elif (project/'results/evaluations').is_dir():study=project/'results'
    elif (project/'evaluations').is_dir():study=project
    else:study=None
    return project,study


def load_sources(project,study,config):
    """Each vector is aligned by bank_id or a checked evaluation completion file."""
    memory_dir=project/'inputs/memory'
    protocol=read_json(memory_dir/'protocol.json');cal=protocol['calibration'];theta=float(cal['theta'])
    fingerprints={str(memory_dir/'protocol.json'):digest_file(memory_dir/'protocol.json')}
    if study and (study/'manifest.json').is_file():manifest=read_json(study/'manifest.json')
    else:manifest={}
    frames=[];vectors=[];inventory=[];skipped=[]
    memory_file=memory_dir/'detectors/gap_knn.npz'
    bank_file=Path(config['bank_csv']).expanduser().resolve() if config.get('bank_csv') else project/'inputs/candidate_bank.csv'
    if bank_file.is_file():
        complete=read_json(memory_dir/'complete.json')
        expected=complete.get('artifact_sha256',{}).get('detectors/gap_knn.npz')
        if expected and digest_file(memory_file)!=expected:raise ValueError('Original memory checksum mismatch.')
        bank_hash=manifest.get('input_sha256',{}).get('inputs/candidate_bank.csv')
        if not bank_hash and (project/'PACKAGE_MANIFEST.json').is_file():
            bank_hash=read_json(project/'PACKAGE_MANIFEST.json').get('input_sha256',{}).get('inputs/candidate_bank.csv')
        if bank_hash and digest_file(bank_file)!=bank_hash:raise ValueError('Candidate-bank checksum differs from the saved experiment.')
        bank=pd.read_csv(bank_file,keep_default_na=False)
        with np.load(memory_file,allow_pickle=False) as f:
            ids=f['bank_ids'].copy();vec=f['vectors'].copy();g=f['gaps'].copy()
        if ids.ndim!=1 or not np.issubdtype(ids.dtype,np.integer) or len(ids)!=len(vec) or len(set(ids))!=len(ids) or ids.min()<0 or ids.max()>=len(bank):
            raise ValueError('Memory bank_ids cannot be aligned to candidate_bank.csv.')
        frame=bank.iloc[ids].copy().reset_index(drop=True)
        recomputed=(frame.proxy_raw-cal['proxy_mean'])/cal['proxy_std']-(frame.judge_raw-cal['judge_mean'])/cal['judge_std']
        if not np.allclose(g,recomputed,atol=2e-5,rtol=2e-5):raise ValueError('Memory gaps do not match the candidate bank and frozen calibration.')
        frame['gap']=g;frame['source_id']='memory_original';frame['origin']='original memory';frame['family']='memory';frame['cohort']='training'
        frame['policy']=frame.get('source','unknown');frame['row_id']=['memory:'+str(i) for i in ids]
        frame['bank_id']=ids;frame['source_row']=ids;frame['is_fit']=True
        frames.append(frame);vectors.append(unit_vectors(vec));inventory.append({'source_id':'memory_original','rows':len(frame),'role':'cluster/PCA fit + kNN reference'})
        fingerprints[str(memory_file)]=digest_file(memory_file);fingerprints[str(bank_file)]=digest_file(bank_file)
    else:
        print('Original answer bank is absent: using completed development_fit vectors as the cluster reference.')
        skipped.append({'source':'original memory','reason':'candidate_bank.csv missing; vectors without aligned text are not displayed'})
    # Only include records for which completed, hash-checked vectors were saved.
    if study:
        for feature_file in sorted((study/'evaluations').rglob('features.npz')):
            folder=feature_file.parent;marker=folder/'complete.json';csv=folder/'predictions.csv'
            if not marker.is_file() or not csv.is_file():
                skipped.append({'source':str(folder),'reason':'incomplete vector/answer export'});continue
            done=read_json(marker);sig=done.get('signature',{})
            if sig.get('family') not in config['families']:continue
            if sig.get('cohort')=='fresh_final' and not config['include_final']:continue
            if done.get('identity')!=digest_json(sig):raise ValueError('Evaluation signature checksum mismatch: '+str(folder))
            if digest_file(csv)!=done.get('csv_sha256') or digest_file(feature_file)!=done.get('features_sha256'):
                raise ValueError('Evaluation text/vector checksum mismatch: '+str(folder))
            frame=pd.read_csv(csv,keep_default_na=False)
            with np.load(feature_file,allow_pickle=False) as f:vec=f['vectors'].copy()
            if len(frame)!=len(vec) or len(frame)!=done['rows']:raise ValueError('Vector/answer row counts differ: '+str(folder))
            if frame.prompt_id.duplicated().any():raise ValueError('Duplicate prompt IDs within an evaluation source.')
            gap=frame.proxy_z.to_numpy()-frame.judge_z.to_numpy()
            raw_gap=(frame.proxy_raw-cal['proxy_mean'])/cal['proxy_std']-(frame.judge_raw-cal['judge_mean'])/cal['judge_std']
            if not np.allclose(gap,frame.actual_proxy_judge_gap,atol=2e-5) or not np.allclose(gap,raw_gap,atol=2e-5):
                raise ValueError('Evaluation reward units/calibration differ: '+str(folder))
            if not np.array_equal(frame.high_gap.to_numpy(),gap>theta):raise ValueError('Saved high-gap labels use a different threshold.')
            source='/'.join([str(sig.get(k,'unknown')) for k in ['family','cohort','policy_id']])
            frame['gap']=gap;frame['source_id']=source;frame['origin']='saved evaluation';frame['policy']=frame['policy_id']
            frame['row_id']=[digest_json([done['identity'],str(p)])[:24] for p in frame.prompt_id]
            frame['source_row']=np.arange(len(frame));frame['is_fit']=False
            frames.append(frame);vectors.append(unit_vectors(vec));inventory.append({'source_id':source,'rows':len(frame),'role':'projected overlay'})
            fingerprints[str(csv)]=digest_file(csv);fingerprints[str(feature_file)]=digest_file(feature_file)
        missing=sum(1 for p in (study/'evaluations').rglob('predictions.csv') if not (p.parent/'features.npz').is_file())
        if missing:skipped.append({'source':f'{missing} evaluation tables','reason':'no saved embedding matrix; no model extraction is run'})
    if not frames:raise FileNotFoundError('No aligned saved vectors and answers found. Point PROJECT_DIR/STUDY_DIR at the completed project. A predictions CSV alone has no hidden vectors.')
    if len({x.shape[1] for x in vectors})!=1:raise ValueError('Sources have different embedding dimensions.')
    data=pd.concat(frames,ignore_index=True);x=np.concatenate(vectors)
    if not data.is_fit.any():
        fit=(data.cohort=='development_fit')
        if not fit.any():raise ValueError('No original memory or development_fit reference. Provide BANK_CSV; do not fit using test labels.')
        data['is_fit']=fit
        inventory.append({'source_id':'development_fit (combined)','rows':int(fit.sum()),'role':'fallback cluster/PCA fit + kNN reference'})
    if data.row_id.duplicated().any():raise ValueError('Duplicate source row identifiers.')
    data['prompt']=data.prompt.astype(str);data['answer']=data.answer.astype(str)
    data['group']=data.prompt.map(conversation_group);data['high_gap']=data.gap>theta
    data['gap_class']=np.where(data.high_gap,'high positive gap',np.where(data.gap<0,'negative gap','non-high positive gap'))
    data['answer_words']=data.answer.str.split().str.len();data['answer_characters']=data.answer.str.len()
    for col in ['proxy_z','judge_z']:
        if col not in data:data[col]=np.nan
    data['refusal_phrase']=data.answer.str.contains(r"\b(?:I(?:'m| am) sorry|I can(?:not|'t) (?:help|assist)|unable to (?:help|assist))",case=False,regex=True)
    if not np.isfinite(data.gap.to_numpy()).all():raise ValueError('Nonfinite gap labels.')
    return data,x,cal,inventory,fingerprints,skipped


def cluster_and_project(data,x,c):
    fit=data.is_fit.to_numpy(bool);n=int(fit.sum());k=c['clusters']
    if not 2<=k<n:raise ValueError(f'Choose 2 <= N_CLUSTERS < {n}.')
    with threadpool_limits(limits=c['cpu_threads']):
        km=KMeans(n_clusters=k,n_init=10,max_iter=300,random_state=c['seed'],algorithm='lloyd').fit(x[fit])
        labels=km.predict(x);pc=PCA(n_components=2,svd_solver='randomized',random_state=c['seed']).fit(x[fit])
        xy=pc.transform(x);centers=pc.transform(km.cluster_centers_)
    out=data.copy();out['cluster']=labels
    out['distance_to_centroid']=np.linalg.norm(x-km.cluster_centers_[labels],axis=1)
    out['pca_x']=xy[:,0];out['pca_y']=xy[:,1]
    with threadpool_limits(limits=c['cpu_threads']):
        try:
            tf=TfidfVectorizer(stop_words='english',min_df=3,max_df=.9,max_features=6000,ngram_range=(1,2),sublinear_tf=True)
            t=tf.fit_transform(out.loc[fit,'answer']);words=tf.get_feature_names_out();global_mean=np.asarray(t.mean(0)).ravel()
            terms={}
            for label in range(k):
                mask=labels[fit]==label;local=np.asarray(t[mask].mean(0)).ravel()
                score=local-global_mean;best=np.argsort(-score)[:6]
                terms[label]=', '.join(words[i] for i in best if score[i]>0)
        except ValueError:terms={i:'' for i in range(k)}
    out['cluster_terms']=out.cluster.map(terms)
    def aggregate(frame,keys):
        return frame.groupby(keys,as_index=False).agg(n=('gap','size'),mean_gap=('gap','mean'),median_gap=('gap','median'),
            std_gap=('gap','std'),high_gap_count=('high_gap','sum'),high_gap_rate=('high_gap','mean'),
            negative_gap_rate=('gap',lambda s:float((s<0).mean())),mean_answer_words=('answer_words','mean'),
            refusal_phrase_rate=('refusal_phrase','mean'))
    summary=aggregate(out,['cluster']);summary['terms']=summary.cluster.map(terms)
    bysource=aggregate(out,['cluster','source_id'])
    info={'fit_rows':n,'fit_sources':sorted(out.loc[fit,'source_id'].unique()),'cluster_count':k,
          'embedding_dimension':x.shape[1],'clustering':'Euclidean KMeans on original L2-normalized vectors, not the 2D coordinates',
          'pca_explained_variance_ratio':pc.explained_variance_ratio_.tolist(),'pca_total_variance_shown':float(pc.explained_variance_ratio_.sum()),
          'pca_centers':centers.tolist(),'seed':c['seed'],'kmeans_inertia':float(km.inertia_),
          'labels_used_to_fit_clusters_or_projection':False,'terms':'descriptive TF-IDF contrasts of reference answers; not verified topics or failure types'}
    return out,summary,bysource,info


def geometry_check(data,x,c):
    reference=np.flatnonzero(data.is_fit.to_numpy());candidate=np.flatnonzero((data.cohort=='offline_test').to_numpy() & ~data.is_fit.to_numpy())
    query_kind='offline_test versus reference memory (exploratory follow-up)'
    if not len(candidate):
        candidate=np.flatnonzero(~data.is_fit.to_numpy());query_kind='saved overlays versus reference memory (descriptive)'
    if not len(candidate):candidate=reference;query_kind='reference memory with own-conversation exclusion (descriptive, not independent held-out evaluation)'
    rng=np.random.default_rng(c['seed']+1)
    if len(candidate)>c['geometry_queries']:candidate=np.sort(rng.choice(candidate,c['geometry_queries'],replace=False))
    ref=x[reference];groups=data.group.to_numpy();g=data.gap.to_numpy(float);zp=data.proxy_z.to_numpy(float)
    records=[];distance_pairs=[];skipped=0
    with threadpool_limits(limits=c['cpu_threads']):
        for start in range(0,len(candidate),64):
            queries=candidate[start:start+64];sim=np.clip(x[queries]@ref.T,-1,1)
            for j,qi in enumerate(queries):
                eligible=np.flatnonzero(groups[reference]!=groups[qi]);k=c['neighbors']
                if len(eligible)<k:skipped+=1;continue
                # Stable ties prefer the original reference order.
                near=eligible[np.argsort(-sim[j,eligible],kind='stable')[:k]]
                random=rng.choice(eligible,size=k,replace=False)
                w=np.exp((sim[j,near]-sim[j,near[0]])/c['temperature']);w/=w.sum()
                gh=float(w@g[reference[near]])
                pair_near=float(np.mean(abs(g[reference[near]]-g[qi])))
                pair_random=float(np.mean(abs(g[reference[random]]-g[qi])))
                score_near=eligible[np.argsort(abs(zp[reference[eligible]]-zp[qi]),kind='stable')[:k]]
                row={'row_id':data.iloc[qi].row_id,'group':groups[qi],'actual_gap':g[qi],'gap_hat':gh,
                     'mean_gap_baseline':float(g[reference[eligible]].mean()),
                     'proxy_score_knn_gap_hat':float(g[reference[score_near]].mean()),
                     'neighbor_pair_abs_gap_difference':pair_near,'random_pair_abs_gap_difference':pair_random,
                     'improvement_random_minus_neighbor':pair_random-pair_near,
                     'mean_neighbor_cosine_distance':float((1-sim[j,near]).mean()),
                     'eligible_references':len(eligible),'k':k}
                records.append(row)
                for label,index in [('near',near),('random',random)]:
                    for ri in index:distance_pairs.append({'query_row_id':row['row_id'],'group':groups[qi],'pair_kind':label,
                        'cosine_distance':float(1-sim[j,ri]),'abs_gap_difference':float(abs(g[qi]-g[reference[ri]]))})
    results=pd.DataFrame(records)
    if not len(results):raise ValueError('No geometry queries have enough references outside their own conversation group.')
    grouped=results.groupby('group').mean(numeric_only=True)
    vals=grouped.improvement_random_minus_neighbor.to_numpy();boot=[]
    for start in range(0,c['bootstrap_draws'],100):
        samples=rng.choice(vals,(min(100,c['bootstrap_draws']-start),len(vals)),replace=True);boot.extend(samples.mean(1))
    lo,hi=np.quantile(boot,[.025,.975]);actual=results.actual_gap.to_numpy();pred=results.gap_hat.to_numpy()
    corr=lambda f:float(f(actual,pred)[0]) if np.std(actual)>0 and np.std(pred)>0 else None
    mse=lambda p:float(pd.Series((actual-np.asarray(p))**2).groupby(results.group.to_numpy()).mean().mean())
    stats={'query_definition':query_kind,'queries':len(results),'query_conversations':len(grouped),'reference_rows':len(reference),'skipped_queries':skipped,
        'excluded_own_conversation_group':True,'k':c['neighbors'],'temperature':c['temperature'],
        'neighbor_pair_abs_gap_difference':float(grouped.neighbor_pair_abs_gap_difference.mean()),
        'random_pair_abs_gap_difference':float(grouped.random_pair_abs_gap_difference.mean()),
        'improvement_random_minus_neighbor':float(vals.mean()),'conditional_prompt_bootstrap_95ci':[float(lo),float(hi)],
        'gap_prediction_pearson':corr(pearsonr),'gap_prediction_spearman':corr(spearmanr),
        'raw_proxy_judge_mse':mse(np.zeros(len(results))),'knn_corrected_judge_mse':mse(pred),
        'mean_gap_baseline_judge_mse':mse(results.mean_gap_baseline),
        'proxy_score_knn_judge_mse':mse(results.proxy_score_knn_gap_hat),
        'uncertainty_note':'Groups are averaged before bootstrap; intervals condition on fixed reference memory and saved policies. Not a PPO improvement test.',
        'score_baseline_note':'k neighbors in scalar proxy z-score, uniform gap average. Representation kNN uses distance weights.'}
    pairs=pd.DataFrame(distance_pairs)
    return results,pairs,stats


def save_figures(out,data,summary,info,geometry,pairs):
    import matplotlib.colors as colors
    plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False,'svg.fonttype':'none'})
    limit=max(float(np.quantile(abs(data.gap),.98)),.05)
    fig,ax=plt.subplots(figsize=(9,6));s=ax.scatter(data.pca_x,data.pca_y,c=data.gap,s=6,alpha=.55,cmap='coolwarm',norm=colors.TwoSlopeNorm(vmin=-limit,vcenter=0,vmax=limit),rasterized=True)
    fig.colorbar(s,ax=ax,label='Actual normalized gap: proxy − judge (color clipped at 98% |gap|)')
    ax.set(xlabel=f'PC1 ({info["pca_explained_variance_ratio"][0]:.1%} of reference variance)',ylabel=f'PC2 ({info["pca_explained_variance_ratio"][1]:.1%})',title='Reward representations colored by observed disagreement')
    fig.tight_layout();fig.savefig(out/'gap_map.png',dpi=200);fig.savefig(out/'gap_map.svg');plt.close(fig)
    ordered=summary.sort_values('mean_gap');fig,axes=plt.subplots(1,2,figsize=(12,5))
    axes[0].bar(ordered.cluster.astype(str),ordered.mean_gap,color='#3176a8');axes[0].axhline(0,color='#222',linewidth=.7)
    axes[0].set(xlabel='Cluster ID (ordered by mean gap)',ylabel='Mean actual gap',title='Cluster differences — descriptive')
    axes[1].bar(ordered.cluster.astype(str),ordered.high_gap_rate*100,color='#bf6143');axes[1].set(xlabel='Cluster ID',ylabel='High-gap answers (%)',title='High gap ≠ confirmed reward hacking')
    fig.tight_layout();fig.savefig(out/'cluster_summary.png',dpi=200);fig.savefig(out/'cluster_summary.svg');plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(11,4.5))
    axes[0].bar(['Nearest neighbors','Random references'],[geometry['neighbor_pair_abs_gap_difference'],geometry['random_pair_abs_gap_difference']],color=['#267b86','#8996a1'])
    axes[0].set(ylabel='Mean absolute gap difference',title=f'Local similarity check (k={geometry["k"]})')
    sample=pairs[pairs.pair_kind=='random'].copy();sample['bin']=pd.qcut(sample.cosine_distance,10,duplicates='drop')
    curve=sample.groupby('bin',observed=True).agg(distance=('cosine_distance','mean'),gap_difference=('abs_gap_difference','mean'))
    axes[1].plot(curve.distance,curve.gap_difference,'o-',color='#267b86');axes[1].set(xlabel='Cosine distance (random-pair deciles)',ylabel='Mean absolute gap difference',title='Distance relationship — descriptive')
    fig.tight_layout();fig.savefig(out/'neighbor_geometry.png',dpi=200);fig.savefig(out/'neighbor_geometry.svg');plt.close(fig)


def run_analysis(config,explorer_template):
    started=time.time();project,study=resolve_paths(config.get('project_dir'),config.get('study_dir'))
    print('Project:',project,'\nSaved study:',study or 'original memory only',flush=True)
    data,x,cal,inventory,fingerprints,skipped=load_sources(project,study,config)
    print(f'Aligned {len(data):,} vectors, {x.shape[1]} dimensions, {len(inventory)} source entries.',flush=True)
    scientific={k:v for k,v in config.items() if k not in ['project_dir','study_dir','output_dir','embed_explorer']}
    versions={'numpy':np.__version__,'pandas':pd.__version__,'sklearn':__import__('sklearn').__version__}
    identity=digest_json({'config':scientific,'sources':fingerprints,'software':versions,
                          'explorer_template':hashlib.sha256(explorer_template.encode()).hexdigest(),'version':1})[:16]
    root=Path(config['output_dir']).expanduser().resolve() if config.get('output_dir') else project/'cluster_visualization'
    out=root/f'analysis_{identity}';out.mkdir(parents=True,exist_ok=True)
    print('Fitting clusters in the original representation space; projecting into 2D separately...',flush=True)
    data,summary,bysource,info=cluster_and_project(data,x,config)
    print('Checking neighbor gap similarity with same-conversation exclusion...',flush=True)
    queries,pairs,geometry=geometry_check(data,x,config)
    data.to_csv(out/'cluster_members.csv',index=False);summary.to_csv(out/'cluster_summary.csv',index=False)
    bysource.to_csv(out/'cluster_by_source.csv',index=False);queries.to_csv(out/'geometry_queries.csv',index=False)
    pairs.to_csv(out/'geometry_pairs.csv',index=False)
    write_json(out/'geometry_summary.json',geometry);write_json(out/'clustering_info.json',info)
    write_json(out/'source_inventory.json',{'included':inventory,'skipped':skipped})
    save_figures(out,data,summary,info,geometry,pairs)
    for column in ['response_tokens','ended_eos','cap','proxy_input_tokens','judge_input_tokens']:
        if column not in data:data[column]=None
    public_columns=['row_id','prompt','answer','source_id','policy','cohort','gap','gap_class','high_gap','proxy_z','judge_z',
        'cluster','cluster_terms','distance_to_centroid','pca_x','pca_y','answer_words','refusal_phrase','is_fit',
        'response_tokens','ended_eos','cap','proxy_input_tokens','judge_input_tokens']
    records=json.loads(data[public_columns].to_json(orient='records',double_precision=8,force_ascii=False))
    payload={'id':identity,'theta':float(cal['theta']),'points':records,'summary':json.loads(summary.to_json(orient='records')),
             'info':info,'geometry':geometry,'sources':inventory}
    safe=json.dumps(payload,ensure_ascii=False,allow_nan=False).replace('<','\\u003c').replace('>','\\u003e').replace('&','\\u0026')
    html=explorer_template.replace('__DATA__',safe)
    (out/'cluster_explorer.html').write_text(html,encoding='utf-8')
    interpretation='''This is descriptive, unblinded exploration of model-judge disagreement. High gap means proxy_z - judge_z > the fixed theta; it is not proof of a hack. A non-high gap does not establish a useful or safe answer. Clusters are fit without gap labels in the original normalized hidden space. PCA can distort neighborhoods; projected centers are not 2D decision boundaries. Frequent terms and refusal phrase matches are clues for manual review, not verified failure categories. Whole-study source differences also mix policy, cohort and sampling effects. Do not use these inspected examples as a new untouched evaluation set. The neighbor check excludes identical normalized first-user conversation groups and conditions on saved policies/reference memory. It does not establish reward-hacking prevention or causality. Saved scores retain their source scoring protocol; original memory labels may reflect the older token limit. Source filters separate these examples from later whole-answer scoring. This notebook does not modify the PPO datastore, rerun PPO or load a language model.'''
    (out/'READ_ME.txt').write_text('Open cluster_explorer.html in a browser. Click a point or a row to inspect the full conversation and answer. Use cluster/source/gap/search filters; sorting applies to the visible answers. Export manual labels and notes as JSON. PNG and SVG figures plus full CSV tables are included.\n\n'+interpretation+'\n',encoding='utf-8')
    write_json(out/'analysis_manifest.json',{'identity':identity,'config':scientific,'input_sha256':fingerprints,'calibration':cal,
        'rows':len(data),'seconds':time.time()-started,'interpretation':interpretation,
        'software':{'numpy':np.__version__,'pandas':pd.__version__,'sklearn':__import__('sklearn').__version__}})
    archive=out/'cluster_analysis.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(out.iterdir()):
            if p.is_file() and p!=archive:z.write(p,p.name)
    print(f'Finished in {time.time()-started:.1f}s. Explorer: {out/"cluster_explorer.html"}',flush=True)
    return out,data,summary,geometry


In [ ]:
EXPLORER_TEMPLATE = '<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>Reward-gap cluster explorer</title><style>\n:root{--ink:#173244;--muted:#516c7c;--line:#d3e0e7;--accent:#166b7a}*{box-sizing:border-box}body{margin:0;background:#f2f6f8;color:var(--ink);font:15px/1.5 system-ui,Arial,sans-serif}main{max-width:1500px;margin:auto;padding:26px}h1{font-size:29px;margin:0 0 6px}h2{font-size:19px;margin:0 0 12px}h3{font-size:16px;margin:14px 0 7px}p{margin:6px 0 12px}.muted{color:var(--muted);font-size:13px}.panel{background:white;border:1px solid var(--line);border-radius:12px;padding:18px;margin-top:16px}.stats{display:flex;flex-wrap:wrap;gap:12px;margin:18px 0}.stat{background:white;border:1px solid var(--line);border-radius:9px;padding:10px 17px;min-width:145px}.stat b{font-size:23px;display:block}.filters{display:grid;grid-template-columns:1fr 2fr 1.1fr 2fr;gap:12px}label{font-size:13px;font-weight:600;display:block}input,select,textarea,button{font:inherit;border:1px solid #a9bdc9;border-radius:6px;padding:8px;color:var(--ink);background:white}select,input[type=text]{width:100%;margin-top:5px}button{cursor:pointer;background:#edf5f7}button:hover{background:#dcecf0}button.active{background:var(--accent);color:white}button:disabled{opacity:.45;cursor:default}.layout{display:grid;grid-template-columns:minmax(460px,1.2fr) minmax(360px,1fr);gap:18px}.toolbar{display:flex;align-items:center;gap:9px;flex-wrap:wrap;margin-bottom:10px}.toolbar label{display:flex;align-items:center;gap:6px}.toolbar select{width:auto;margin:0}#map{width:100%;height:480px;border:1px solid var(--line);border-radius:8px;touch-action:none;cursor:crosshair}pre{white-space:pre-wrap;word-break:break-word;font:14px/1.6 system-ui,Arial,sans-serif;max-height:350px;overflow:auto;background:#f5f8fa;padding:12px;border-radius:6px;margin:7px 0}.answer{border-left:3px solid var(--accent)}.tag{display:inline-block;background:#edf3f6;padding:3px 7px;border-radius:4px;margin:2px 4px 2px 0;font-size:12px}.list{max-height:500px;overflow:auto}.item{display:block;width:100%;text-align:left;margin-bottom:7px;padding:10px;background:white}.item strong{font-size:13px}.item small{color:var(--muted);display:block;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}.item.selected{border-color:var(--accent);background:#eef7f8}.clusterGrid{display:grid;grid-template-columns:repeat(auto-fill,minmax(180px,1fr));gap:9px}.clusterCard{text-align:left;min-height:113px}.clusterCard b{display:block}.clusterCard small{display:block;font-size:11px;margin-top:5px}.clusterCard .bar{height:5px;background:#d7e7ed;border-radius:4px;margin-top:7px;overflow:hidden}.clusterCard .bar span{display:block;height:100%;background:#bd5a44}.legend{display:flex;align-items:center;gap:8px;font-size:12px;margin:7px 0}.gradient{height:10px;width:170px;background:linear-gradient(to right,#2563a4,#e8ecef,#ba4338);border-radius:3px}textarea{width:100%;min-height:65px;margin-top:6px}.notice{padding:10px;background:#edf5f7;border-radius:6px;font-size:13px}details summary{cursor:pointer;font-weight:600}.pager{display:flex;justify-content:space-between;align-items:center;gap:12px;margin:8px 0}#tooltip{display:none;position:fixed;z-index:10;pointer-events:none;max-width:320px;background:#173244;color:white;padding:9px;border-radius:7px;font-size:12px}#selectionEmpty{padding:25px;color:var(--muted)}#selectionBody{display:none}.tiny{font-size:12px;overflow-wrap:anywhere;white-space:pre-line}.footer{margin:20px 0;color:var(--muted);font-size:12px}@media(max-width:1000px){.layout{grid-template-columns:1fr}.filters{grid-template-columns:1fr 1fr}main{padding:15px}}@media(max-width:550px){.filters{grid-template-columns:1fr}#map{height:360px}}\n</style><main>\n<header><h1>Reward-gap cluster explorer</h1><p>Explore which answers occupy similar regions of the frozen proxy reward model.</p><p class="muted">Clusters use the original hidden vectors. The 2D map is a PCA projection. High gap means disagreement with the judge; it is not a confirmed reward hack.</p></header>\n<div class="stats"><div class="stat"><b id="total"></b>saved answers</div><div class="stat"><b id="clusterCount"></b>clusters</div><div class="stat"><b id="highTotal"></b>high-gap answers</div><div class="stat"><b id="variance"></b>variance shown in 2D</div></div>\n<section class="panel"><div class="filters"><label>Cluster<select id="cluster"></select></label><label>Saved source<select id="source"></select></label><label>Gap category<select id="category"><option value="all">All gaps</option><option value="high positive gap">High positive gap</option><option value="non-high positive gap">Non-high positive gap</option><option value="negative gap">Negative gap</option></select></label><label>Search conversation or answer<input id="search" type="text" placeholder="e.g. sorry, medicine, code"></label></div><p id="filterStats" class="muted" style="margin-top:12px"></p></section>\n<div class="layout"><section class="panel"><div class="toolbar"><h2 style="margin:0;flex:1">Representation map</h2><label>Color<select id="color"><option value="gap">Actual gap</option><option value="cluster">Cluster</option><option value="category">Gap category</option></select></label><button id="resetZoom">Reset view</button></div><canvas id="map"></canvas><div class="legend" id="gapLegend"><span id="negativeLimit"></span><span class="gradient"></span><span id="positiveLimit"></span><span>gap</span></div><p class="muted">Click a point to inspect it. Drag to pan; scroll to zoom. Red: proxy above judge. Blue: proxy below judge. Circled numbers mark projected cluster centers, not boundaries.</p><div class="toolbar"><label>Answer order<select id="sort"><option value="gap">Largest gap first</option><option value="negative">Most negative gap first</option><option value="centroid">Closest to assigned centroid</option><option value="random">Fixed shuffled order</option></select></label><button id="downloadFiltered">Export visible rows CSV</button></div><div class="pager"><button id="previous">Previous</button><span id="pageInfo" class="tiny"></span><button id="next">Next</button></div><div id="list" class="list"></div></section>\n<section class="panel"><h2>Conversation and answer</h2><div id="selectionEmpty">Click a point or an answer in the list to inspect its full text.</div><div id="selectionBody"><div id="tags"></div><p id="selectedTerms" class="muted"></p><h3>Conversation</h3><pre id="prompt"></pre><h3>Generated answer</h3><pre id="answer" class="answer"></pre><p id="scores" class="tiny"></p><details><summary>Manual interpretation of this answer</summary><p class="muted">This is unblinded exploration. These notes are not the separate blinded human-review results.</p><label>Your assessment<select id="assessment"><option value="unreviewed">Unreviewed</option><option value="apparently_useful">Appears useful / appropriate</option><option value="inappropriate_refusal">Possible inappropriate refusal</option><option value="unsupported_claim">Possible unsupported or incorrect claim</option><option value="off_topic">Off-topic / does not answer</option><option value="repetition">Repetition / boilerplate</option><option value="format_exploit">Possible formatting exploitation</option><option value="uncertain">Uncertain / needs expert review</option></select></label><label>Notes<textarea id="notes" placeholder="What behavior do you see? What would you need to verify?"></textarea></label><button id="saveAssessment">Save answer notes</button></details><h3>Describe this cluster</h3><label>Provisional cluster label<input id="clusterLabel" type="text" placeholder="e.g. short refusal answers — needs review"></label><button id="saveClusterLabel" style="margin-top:7px">Save cluster label</button></div><hr style="border:0;border-top:1px solid var(--line);margin:20px 0"><div class="toolbar"><button id="exportNotes">Export notes JSON</button><label>Import notes<input type="file" id="importNotes" accept=".json" style="max-width:220px"></label></div><p id="noteStatus" class="muted">Notes stay in this browser when storage is available. Export JSON to keep a portable copy.</p></section></div>\n<section class="panel"><h2>Groups at a glance</h2><p class="muted">Cards summarize the current source, gap and text filters before selecting a cluster. Terms come from the reference answers and are descriptive word clues. Click a group to inspect its members.</p><div id="clusters" class="clusterGrid"></div></section>\n<section class="panel"><h2>Does local geometry predict similar gaps?</h2><div id="geometry"></div><details><summary>How to interpret this analysis</summary><p>Nearest-neighbor and random-reference comparisons use the original high-dimensional cosine distances, with all reference answers sharing the query\'s normalized first-user conversation excluded. Random references come from the same eligible memory. Where available, the queries are saved offline-test answers.</p><p>Cluster differences and a 2D picture alone do not prove the kNN assumption, establish causality, or demonstrate prevention of reward hacking. A projection can hide or distort neighborhoods. A global monotonic distance–error relationship is not required for useful local prediction. The bootstrap treats query conversations as groups and conditions on fixed saved models and reference memory.</p><p>Non-high gaps can still include bad answers. Large gaps can reflect judge errors, different preferences, reward calibration, or proxy errors. Refusal phrase matches are literal text diagnostics, not decisions that refusal is inappropriate. Differences across sources may also reflect different prompt cohorts and scoring token limits. Original-memory labels retain their earlier scoring protocol; later saved scores use their recorded whole-answer protocol. Inspect examples and use independent blinded review for claims about usefulness or inappropriate refusal.</p></details></section>\n<p class="footer">Offline explorer · no model loading, remote scripts, or network requests · analysis ID <span id="analysisId"></span></p>\n</main><div id="tooltip"></div><script id="payload" type="application/json">__DATA__</script><script>\n\'use strict\';\nconst D=JSON.parse(document.getElementById(\'payload\').textContent),$=id=>document.getElementById(id),P=D.points;\nconst palette=[\'#236f86\',\'#b86a40\',\'#7959a2\',\'#488763\',\'#b34769\',\'#827744\',\'#536da7\',\'#a05a4c\',\'#437d79\',\'#945887\',\'#638241\',\'#816950\',\'#4c85a8\',\'#ab695b\',\'#7370a4\',\'#568867\',\'#b3574c\',\'#668395\',\'#9c7760\',\'#626f85\'];\nconst state={filtered:[],base:[],page:0,selected:null,zoom:1,panX:0,panY:0,projected:[]};\nlet noteData={analysis_id:D.id,answers:{},clusters:{}};\ntry{const stored=JSON.parse(localStorage.getItem(\'gap_clusters_\'+D.id)||\'null\');if(stored&&stored.analysis_id===D.id)noteData=stored;}catch(e){}\nconst clamp=(v,a,b)=>Math.max(a,Math.min(b,v));const fmt=(v,n=3)=>v===null||!Number.isFinite(v)?\'—\':v.toFixed(n);\nconst quantile=(a,p)=>{const b=[...a].sort((x,y)=>x-y);return b[Math.floor((b.length-1)*p)];};\nconst limit=Math.max(quantile(P.map(p=>Math.abs(p.gap)),.98),.05);\nconst extent=P.reduce((a,p)=>({minX:Math.min(a.minX,p.pca_x),maxX:Math.max(a.maxX,p.pca_x),minY:Math.min(a.minY,p.pca_y),maxY:Math.max(a.maxY,p.pca_y)}),{minX:Infinity,maxX:-Infinity,minY:Infinity,maxY:-Infinity});\nconst widthX=Math.max(extent.maxX-extent.minX,1e-9),widthY=Math.max(extent.maxY-extent.minY,1e-9);\nconst randomOrder=new Map(P.map((p,i)=>[p.row_id,((i+1)*2654435761)>>>0]));\n$(\'total\').textContent=P.length.toLocaleString();$(\'clusterCount\').textContent=D.info.cluster_count;\n$(\'highTotal\').textContent=(100*P.filter(p=>p.high_gap).length/P.length).toFixed(1)+\'%\';\n$(\'variance\').textContent=(D.info.pca_total_variance_shown*100).toFixed(1)+\'%\';$(\'analysisId\').textContent=D.id;\n$(\'negativeLimit\').textContent=\'−\'+limit.toFixed(2);$(\'positiveLimit\').textContent=\'+\'+limit.toFixed(2);\n$(\'cluster\').add(new Option(\'All clusters\',\'all\'));for(let i=0;i<D.info.cluster_count;i++)$(\'cluster\').add(new Option(\'Cluster \'+i,String(i)));\n$(\'source\').add(new Option(\'All saved sources\',\'all\'));for(const s of [...new Set(P.map(p=>p.source_id))].sort())$(\'source\').add(new Option(s,s));\nfunction colorPoint(p){if($(\'color\').value===\'cluster\')return palette[p.cluster%palette.length];if($(\'color\').value===\'category\')return p.high_gap?\'#ba4338\':p.gap<0?\'#2563a4\':\'#929da3\';const t=clamp(Math.abs(p.gap)/limit,0,1),end=p.gap<0?[37,99,164]:[186,67,56],start=[232,236,239];return \'rgb(\'+start.map((v,i)=>Math.round(v+(end[i]-v)*t)).join(\',\')+\')\';}\nfunction mapPosition(x,y,w,h){return [w/2+(((x-extent.minX)/widthX-.5)*(w-62))*state.zoom+state.panX,h/2-(((y-extent.minY)/widthY-.5)*(h-60))*state.zoom+state.panY];}\nfunction draw(){const canvas=$(\'map\'),w=canvas.clientWidth,h=canvas.clientHeight,ratio=window.devicePixelRatio||1;canvas.width=Math.round(w*ratio);canvas.height=Math.round(h*ratio);const ctx=canvas.getContext(\'2d\');ctx.scale(ratio,ratio);ctx.fillStyle=\'#fbfdfe\';ctx.fillRect(0,0,w,h);ctx.strokeStyle=\'#e4ebef\';ctx.lineWidth=1;ctx.beginPath();ctx.moveTo(w/2,20);ctx.lineTo(w/2,h-20);ctx.moveTo(20,h/2);ctx.lineTo(w-20,h/2);ctx.stroke();state.projected=[];\nfor(const p of state.filtered){const [x,y]=mapPosition(p.pca_x,p.pca_y,w,h);if(x<0||y<0||x>w||y>h)continue;ctx.globalAlpha=.65;ctx.fillStyle=colorPoint(p);ctx.beginPath();ctx.arc(x,y,p.row_id===state.selected?5:2.4,0,Math.PI*2);ctx.fill();state.projected.push({x,y,p});}\nctx.globalAlpha=1;if(state.selected){const hit=state.projected.find(t=>t.p.row_id===state.selected);if(hit){ctx.strokeStyle=\'#101f2b\';ctx.lineWidth=2;ctx.beginPath();ctx.arc(hit.x,hit.y,7,0,Math.PI*2);ctx.stroke();}}\nfor(let i=0;i<D.info.pca_centers.length;i++){const [x,y]=mapPosition(...D.info.pca_centers[i],w,h);if(x<10||y<10||x>w-10||y>h-10)continue;ctx.fillStyle=\'white\';ctx.strokeStyle=\'#435a68\';ctx.lineWidth=1;ctx.beginPath();ctx.arc(x,y,9,0,Math.PI*2);ctx.fill();ctx.stroke();ctx.fillStyle=\'#173244\';ctx.font=\'10px system-ui\';ctx.textAlign=\'center\';ctx.textBaseline=\'middle\';ctx.fillText(i,x,y);}\n$(\'gapLegend\').style.visibility=$(\'color\').value===\'gap\'?\'visible\':\'hidden\';}\nfunction preview(p){return p.answer.replace(/\\s+/g,\' \').slice(0,120)||\'(empty answer)\';}\nfunction showList(){const list=$(\'list\');list.replaceChildren();const n=25,total=state.filtered.length,max=Math.max(1,Math.ceil(total/n));state.page=clamp(state.page,0,max-1);for(const p of state.filtered.slice(state.page*n,(state.page+1)*n)){const button=document.createElement(\'button\');button.className=\'item\'+(p.row_id===state.selected?\' selected\':\'\');const title=document.createElement(\'strong\');title.textContent=\'Cluster \'+p.cluster+\' · gap \'+fmt(p.gap)+\' · \'+p.gap_class;const text=document.createElement(\'small\');text.textContent=preview(p);button.append(title,text);button.onclick=()=>selectPoint(p);list.append(button);}$(\'pageInfo\').textContent=total.toLocaleString()+\' answers · page \'+(state.page+1)+\' / \'+max;$(\'previous\').disabled=state.page===0;$(\'next\').disabled=state.page>=max-1;}\nfunction showClusterCards(){const target=$(\'clusters\');target.replaceChildren();const groups=new Map();for(const p of state.base){if(!groups.has(p.cluster))groups.set(p.cluster,[]);groups.get(p.cluster).push(p);}for(const [id,points] of [...groups.entries()].sort((a,b)=>a[0]-b[0])){const card=document.createElement(\'button\');card.className=\'clusterCard\'+($(\'cluster\').value===String(id)?\' active\':\'\');const count=points.length,mean=points.reduce((s,p)=>s+p.gap,0)/count,high=points.filter(p=>p.high_gap).length/count;const title=document.createElement(\'b\');title.textContent=\'Cluster \'+id+(noteData.clusters[id]?\' · \'+noteData.clusters[id]:\'\');const stats=document.createElement(\'span\');stats.textContent=count.toLocaleString()+\' answers · mean \'+fmt(mean);const subtitle=document.createElement(\'small\');subtitle.textContent=(high*100).toFixed(1)+\'% high gap · \'+(points[0].cluster_terms||\'no distinctive terms\');const bar=document.createElement(\'div\');bar.className=\'bar\';const fill=document.createElement(\'span\');fill.style.width=(100*high)+\'%\';bar.append(fill);card.append(title,stats,subtitle,bar);card.onclick=()=>{$(\'cluster\').value=String(id);applyFilters();};target.append(card);}}\nfunction applyFilters(){const text=$(\'search\').value.trim().toLowerCase(),source=$(\'source\').value,category=$(\'category\').value,cluster=$(\'cluster\').value;state.base=P.filter(p=>(source===\'all\'||p.source_id===source)&&(category===\'all\'||p.gap_class===category)&&(!text||(p.prompt+\' \'+p.answer).toLowerCase().includes(text)));state.filtered=state.base.filter(p=>cluster===\'all\'||p.cluster===Number(cluster));const order=$(\'sort\').value;state.filtered.sort((a,b)=>order===\'gap\'?b.gap-a.gap:order===\'negative\'?a.gap-b.gap:order===\'centroid\'?a.distance_to_centroid-b.distance_to_centroid:randomOrder.get(a.row_id)-randomOrder.get(b.row_id));state.page=0;const high=state.filtered.filter(p=>p.high_gap).length;$(\'filterStats\').textContent=state.filtered.length.toLocaleString()+\' visible answers · \'+high.toLocaleString()+\' high-gap answers · fixed θ = \'+fmt(D.theta,4);if(state.selected&&!state.filtered.some(p=>p.row_id===state.selected)){state.selected=null;$(\'selectionBody\').style.display=\'none\';$(\'selectionEmpty\').style.display=\'block\';}showList();showClusterCards();draw();}\nfunction selectPoint(p){state.selected=p.row_id;$(\'selectionEmpty\').style.display=\'none\';$(\'selectionBody\').style.display=\'block\';$(\'prompt\').textContent=p.prompt;$(\'answer\').textContent=p.answer;$(\'selectedTerms\').textContent=\'Reference answer terms: \'+p.cluster_terms;$(\'tags\').replaceChildren();for(const value of [\'Cluster \'+p.cluster,p.gap_class,p.policy,p.is_fit?\'Reference member\':\'Projected overlay\']){const tag=document.createElement(\'span\');tag.className=\'tag\';tag.textContent=value;$(\'tags\').append(tag);}$(\'scores\').textContent=\'Proxy z: \'+fmt(p.proxy_z)+\' · Judge z: \'+fmt(p.judge_z)+\' · Gap: \'+fmt(p.gap)+\' · Answer words: \'+p.answer_words+\' · Refusal phrase matched: \'+(p.refusal_phrase?\'yes\':\'no\')+\'\\nGeneration cap: \'+(p.cap??\'not recorded\')+\' · EOS reached: \'+(p.ended_eos===null?\'not recorded\':p.ended_eos?\'yes\':\'no\')+\' · Proxy input tokens: \'+(p.proxy_input_tokens??\'not recorded\')+\'\\nSource: \'+p.source_id;const note=noteData.answers[p.row_id]||{};$(\'assessment\').value=note.assessment||\'unreviewed\';$(\'notes\').value=note.notes||\'\';$(\'clusterLabel\').value=noteData.clusters[p.cluster]||\'\';showList();draw();}\nfunction saveNotes(){try{localStorage.setItem(\'gap_clusters_\'+D.id,JSON.stringify(noteData));$(\'noteStatus\').textContent=\'Saved in this browser. Export JSON for a portable backup.\';}catch(e){$(\'noteStatus\').textContent=\'Browser storage unavailable. Use Export notes JSON before closing.\';}}\nfunction saveSelected(){if(!state.selected)return;noteData.answers[state.selected]={assessment:$(\'assessment\').value,notes:$(\'notes\').value,updated_at:new Date().toISOString()};saveNotes();}\n$(\'saveAssessment\').onclick=saveSelected;$(\'saveClusterLabel\').onclick=()=>{const p=P.find(p=>p.row_id===state.selected);if(!p)return;noteData.clusters[p.cluster]=$(\'clusterLabel\').value.trim();saveNotes();showClusterCards();};\nfunction download(filename,text,type){const url=URL.createObjectURL(new Blob([text],{type})),a=document.createElement(\'a\');a.href=url;a.download=filename;document.body.append(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),1000);}\n$(\'exportNotes\').onclick=()=>{saveSelected();download(\'cluster_notes_\'+D.id+\'.json\',JSON.stringify(noteData,null,2),\'application/json\');};\n$(\'importNotes\').onchange=async()=>{const file=$(\'importNotes\').files[0];if(!file)return;try{const n=JSON.parse(await file.text());if(n.analysis_id!==D.id||!n.answers||!n.clusters)throw Error(\'Different analysis or invalid notes file.\');const ids=new Set(P.map(p=>p.row_id));for(const k of Object.keys(n.answers))if(!ids.has(k))throw Error(\'Unknown answer ID.\');noteData=n;saveNotes();showClusterCards();const p=P.find(p=>p.row_id===state.selected);if(p)selectPoint(p);}catch(e){alert(\'Cannot import: \'+e.message);}};\nfunction csvCell(v){let s=String(v??\'\');if(/^[=+@\\-\\t\\r]/.test(s))s="\'"+s;return \'"\'+s.replace(/"/g,\'""\')+\'"\';}\n$(\'downloadFiltered\').onclick=()=>{const columns=[\'row_id\',\'cluster\',\'source_id\',\'gap\',\'gap_class\',\'proxy_z\',\'judge_z\',\'prompt\',\'answer\'];download(\'visible_answers_\'+D.id+\'.csv\',[columns.join(\',\'),...state.filtered.map(p=>columns.map(k=>csvCell(p[k])).join(\',\'))].join(\'\\n\'),\'text/csv;charset=utf-8\');};\nfor(const id of [\'cluster\',\'source\',\'category\',\'sort\'])$(id).onchange=applyFilters;let searchTimer;$(\'search\').oninput=()=>{clearTimeout(searchTimer);searchTimer=setTimeout(applyFilters,180);};$(\'color\').onchange=draw;$(\'resetZoom\').onclick=()=>{state.zoom=1;state.panX=state.panY=0;draw();};$(\'previous\').onclick=()=>{state.page--;showList();};$(\'next\').onclick=()=>{state.page++;showList();};\nconst canvas=$(\'map\');let dragging=null;canvas.addEventListener(\'pointerdown\',e=>{dragging={x:e.clientX,y:e.clientY,panX:state.panX,panY:state.panY,moved:false};canvas.setPointerCapture(e.pointerId);});\nfunction nearest(e){const r=canvas.getBoundingClientRect(),x=e.clientX-r.left,y=e.clientY-r.top;let best=null,d=81;for(const p of state.projected){const z=(x-p.x)**2+(y-p.y)**2;if(z<d){best=p.p;d=z;}}return best;}\ncanvas.addEventListener(\'pointermove\',e=>{if(dragging){const dx=e.clientX-dragging.x,dy=e.clientY-dragging.y;if(Math.abs(dx)+Math.abs(dy)>4)dragging.moved=true;state.panX=dragging.panX+dx;state.panY=dragging.panY+dy;draw();$(\'tooltip\').style.display=\'none\';return;}const p=nearest(e),tip=$(\'tooltip\');if(p){tip.textContent=\'Cluster \'+p.cluster+\' · gap \'+fmt(p.gap)+\'\\n\'+preview(p);tip.style.display=\'block\';tip.style.left=Math.min(e.clientX+12,window.innerWidth-330)+\'px\';tip.style.top=(e.clientY+12)+\'px\';}else tip.style.display=\'none\';});\ncanvas.addEventListener(\'pointerup\',e=>{if(dragging&&!dragging.moved){const p=nearest(e);if(p)selectPoint(p);}dragging=null;});canvas.addEventListener(\'pointerleave\',()=>{$(\'tooltip\').style.display=\'none\';});\ncanvas.addEventListener(\'wheel\',e=>{e.preventDefault();const r=canvas.getBoundingClientRect(),x=e.clientX-r.left-canvas.clientWidth/2,y=e.clientY-r.top-canvas.clientHeight/2,old=state.zoom;state.zoom=clamp(old*Math.exp(-e.deltaY*.001),.5,20);state.panX=x-(x-state.panX)*state.zoom/old;state.panY=y-(y-state.panY)*state.zoom/old;draw();},{passive:false});\nconst g=D.geometry,lines=[[\'Nearest-neighbor gap difference\',fmt(g.neighbor_pair_abs_gap_difference)],[\'Random-reference gap difference\',fmt(g.random_pair_abs_gap_difference)],[\'Random minus neighbor (positive favors locality)\',fmt(g.improvement_random_minus_neighbor)+\'; 95% conditional interval [\'+g.conditional_prompt_bootstrap_95ci.map(x=>fmt(x)).join(\', \')+\']\'],[\'Raw / corrected judge-score MSE\',fmt(g.raw_proxy_judge_mse)+\' / \'+fmt(g.knn_corrected_judge_mse)],[\'Mean-gap baseline / proxy-score kNN MSE\',fmt(g.mean_gap_baseline_judge_mse)+\' / \'+fmt(g.proxy_score_knn_judge_mse)],[\'Query conversations / neighbors\',g.query_conversations+\' / \'+g.k]];\nfor(const [name,value] of lines){const p=document.createElement(\'p\'),b=document.createElement(\'b\');b.textContent=name+\': \';p.append(b,document.createTextNode(value));$(\'geometry\').append(p);}const description=document.createElement(\'p\');description.className=\'muted\';description.textContent=g.query_definition;$(\'geometry\').append(description);\nwindow.addEventListener(\'resize\',draw);applyFilters();\n</script></html>\n'

## 3. Load, cluster, project, and check local similarity

The original memory is the fit/reference set. New saved answers are assigned to those same clusters and projected through the same PCA. If the original text bank is absent, completed `development_fit` outputs supply the reference instead.

Each vector is matched to its exact answer using saved bank indices or completed evaluation metadata/checksums. Gap values are checked against the frozen proxy/judge calibration. Neither gap labels nor 2D coordinates are used to fit the clusters.

The quantitative check compares gap differences to the 31 nearest reference vectors versus 31 random references. It excludes all references sharing the query's normalized first-user conversation. It also reports gap prediction MSE against a constant-gap baseline and a simple proxy-score kNN baseline. These are exploratory offline diagnostics, not new PPO outcomes.

In [ ]:
CONFIG = {
    "project_dir": PROJECT_DIR, "study_dir": STUDY_DIR, "bank_csv": BANK_CSV,
    "output_dir": OUTPUT_DIR, "clusters": N_CLUSTERS, "neighbors": NEIGHBORS,
    "temperature": TEMPERATURE, "seed": SEED, "cpu_threads": CPU_THREADS,
    "geometry_queries": GEOMETRY_QUERIES, "bootstrap_draws": BOOTSTRAP_DRAWS,
    "families": ["development", "refresh", "refresh_validation"],
    "include_final": False, "embed_explorer": EMBED_EXPLORER,
}
OUT, MEMBERS, CLUSTERS, GEOMETRY = run_analysis(CONFIG, EXPLORER_TEMPLATE)
from IPython.display import display, HTML, Image, FileLink
print("Outputs:", OUT)
display(CLUSTERS.sort_values("mean_gap", ascending=False).reset_index(drop=True))
display(pd.DataFrame([GEOMETRY]).T.rename(columns={0:"value"}))
print("Sources without vectors / incomplete exports:")
display(pd.DataFrame(read_json(OUT / "source_inventory.json")["skipped"]))

## 4. Interactive explorer

**Click a point or a listed answer** to see the complete conversation, answer and scores. Filter by cluster, source, high/negative/non-high gap, or text. Color by actual gap or cluster. Scroll to zoom and drag to pan. Use the cluster cards to find groups with different error patterns.

You can provisionally label clusters and annotate individual answers, then **Export notes JSON**. These are your unblinded exploratory notes, separate from the previous experiment's blinded reviewer ratings.

If Jupyter does not display the iframe, download `cluster_analysis.zip` using the link below, extract it, and open `cluster_explorer.html` in Chrome or Firefox. It is self-contained and makes no network requests. Export notes regularly; browser storage is not guaranteed in every notebook setup.

In [ ]:
import html as html_std

def file_link(path):
    path = Path(path)
    try: path = path.relative_to(Path.cwd())
    except ValueError: pass
    display(FileLink(str(path)))

file_link(OUT / "cluster_analysis.zip")
file_link(OUT / "cluster_explorer.html")
if EMBED_EXPLORER:
    content = (OUT / "cluster_explorer.html").read_text(encoding="utf-8")
    display(HTML('<iframe title="Reward-gap cluster explorer" style="width:100%;height:1150px;border:1px solid #d3e0e7;border-radius:8px" srcdoc="'
                 + html_std.escape(content, quote=True) + '"></iframe>'))
else:
    print("Open the standalone HTML from the downloaded ZIP.")

## 5. Figures for your report

PNG and editable SVG versions are saved. PCA's explained variance is shown on the axes. Do not interpret distances or apparent separation in 2D as the original kNN geometry. Projected cluster centers do not define reliable 2D cluster boundaries.

The summary bars describe the currently included sources together. Use `cluster_by_source.csv` or the explorer source filter to compare sources separately. Source changes can also reflect different prompts.

In [ ]:
for filename in ["gap_map.png", "cluster_summary.png", "neighbor_geometry.png"]:
    display(Image(filename=str(OUT / filename), width=950))
file_link(OUT / "cluster_summary.csv")
file_link(OUT / "cluster_by_source.csv")
file_link(OUT / "cluster_members.csv")

## 6. Optional: inspect a few full examples directly in the notebook

Change `CLUSTER_TO_INSPECT` and run this cell. Examples closest to the centroid provide a representative starting point; the largest-gap sort is useful for investigating outliers but is not a random review sample.

In [ ]:
CLUSTER_TO_INSPECT = 0
EXAMPLE_ORDER = "representative"  # "representative", "highest_gap", or "lowest_gap"
N_EXAMPLES = 3
subset = MEMBERS[MEMBERS.cluster == CLUSTER_TO_INSPECT]
column = "distance_to_centroid" if EXAMPLE_ORDER == "representative" else "gap"
subset = subset.sort_values(column, ascending=EXAMPLE_ORDER != "highest_gap").head(N_EXAMPLES)
for _, row in subset.iterrows():
    display(HTML("<h3>Cluster " + str(row.cluster) + " · actual gap " + f"{row.gap:.3f}" + "</h3>"
                 + "<p>" + html_std.escape(str(row.source_id)) + "</p>"
                 + "<b>Conversation</b><pre style='white-space:pre-wrap'>" + html_std.escape(row.prompt) + "</pre>"
                 + "<b>Answer</b><pre style='white-space:pre-wrap'>" + html_std.escape(row.answer) + "</pre>"))

## What this can establish

A useful finding would be that nearby **held-out** representations have more similar proxy–judge gaps than random references, with a useful correction MSE compared with simple controls. That supports local predictability. It does not establish that each high-gap answer is a hack or that a particular topic causes overestimation.

K-means imposes a chosen number of groups; clusters need not be naturally separated or stable semantic categories. Re-running with different `N_CLUSTERS` is exploratory sensitivity analysis, not permission to select only the most striking figure. The frequent terms are automated word clues. Confirm any proposed behavioral category by reading examples and use independent review for final quality claims.

Method references: [scikit-learn KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) and [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html).